# Dose-response study: Oxford-IIIT Pet with ResNet-18

Trains 18 ResNet-18 models from scratch on cat vs dog photos (6 reliance doses x
3 seeds, 12x12 patch shortcut, 30 epochs, 128x128 images), scores them with every
metric and control map, and packs the results into `pet_results.zip` (in the
notebook's Output) to copy back to the laptop. The images come preprocessed in
the bundle (`data/pet128/*.npz`), so nothing is downloaded except Captum.

Before running: **Settings → Accelerator → GPU T4** (or P100), and **Internet on**
(needed once, to install Captum). Then use **Save Version → Save & Run All (Commit)**
so it keeps running with the browser closed.

In [ ]:
# 1. Copy the uploaded project into the writable working folder
import glob, os, shutil, subprocess, sys
found = glob.glob('/kaggle/input/**/dose_response.py', recursive=True)
assert found, 'Project files not found: add your uploaded dataset with "+ Add Input".'
src = os.path.dirname(found[0])
WORK = '/kaggle/working/project'
if not os.path.exists(WORK):
    shutil.copytree(src, WORK)
os.chdir(WORK)
print('project copied from', src)
print(sorted(os.listdir(WORK)))

In [ ]:
# 2. Install the one missing package, and check the GPU
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'captum==0.9.0'], check=True)
import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'No GPU: turn on Settings → Accelerator → GPU, then restart.'
for name in ('trainval.npz', 'test.npz'):
    assert os.path.exists(f'data/pet128/{name}'), f'data/pet128/{name} missing: rebuild the bundle with -Dataset pet.'
print('Pet data found')

In [ ]:
# 3. Train (prints validation accuracy per epoch and minutes per model).
#    Re-running skips models that already finished.
!python -u -W ignore dose_response.py train --bases pet_resnet18 --shortcuts patch --doses -0.5 -0.4 -0.3 -0.2 -0.1 0 --seeds 42 1 2

In [ ]:
# 4. Score every model with all metrics, oracle maps and controls
!python -u -W ignore dose_response.py evaluate --bases pet_resnet18 --shortcuts patch --doses -0.5 -0.4 -0.3 -0.2 -0.1 0 --seeds 42 1 2
!python -u -W ignore dose_response.py placebo --bases pet_resnet18 --shortcuts patch --doses -0.5 -0.4 -0.3 -0.2 -0.1 0 --seeds 42 1 2

In [ ]:
# 5. Pack the results to download
import zipfile
out = '/kaggle/working/pet_results.zip'
files = glob.glob('checkpoint/dose/pet_resnet18_*') + glob.glob('results/dose_response/per_model/pet_resnet18_*')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, f)  # keeps checkpoint/... and results/... paths
print(len(files), 'files ->', out)
shutil.rmtree(WORK)  # keep only the zip in the Output